#### Extract egyption drugs from source

In [17]:
import os
import pandas as pd
import requests
from datetime import datetime

In [18]:
# retrieve the current date and time
BASE_DIR = os.getcwd()
if 'etl' in BASE_DIR:
    BASE_DIR = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(BASE_DIR, 'data', 'raw')
os.makedirs(DATA_DIR, exist_ok=True)

In [19]:
# download the CSV file from the URL and save it to the local data directory
csv_url = "https://raw.githubusercontent.com/karem505/egyptian-drug-database/main/data/egyptian-drugs.csv"
local_file_path = os.path.join(DATA_DIR, 'egyptian_drugs.csv')

In [20]:
# check if the file already exists locally
if not os.path.exists(local_file_path):
    print("loading the CSV file from the URL and saving it to the local data directory...")
    response = requests.get(csv_url, timeout=30)
    response.raise_for_status()
    
    with open(local_file_path, 'wb') as f:
        f.write(response.content)
    print("✅ The CSV file has been downloaded and saved successfully in the data/raw/ directory!")
else:
    print("📁 The file already exists locally in the data/raw/ directory, reading it directly...")

📁 The file already exists locally in the data/raw/ directory, reading it directly...


In [21]:
df_local = pd.read_csv(local_file_path, encoding='utf-8')

display(df_local.head(10))

,commercial_name_en,commercial_name_ar,scientific_name,manufacturer,drug_class,route,price_egp
0,1 2 3 (ONE TWO THREE) 20 F.C.TABS.,1 2 3,CHLORPHENIRAMINE+PARACETAMOL(ACETAMINOPHEN)+PS...,HIKMA PHARMA,COLD PRODUCTS,ORAL.SOLID,10.0
1,1 2 3 (ONE TWO THREE) EXTRA 20 F.C.TABS.,1 2 3 إكسترا,CHLORPHENIRAMINE+PARACETAMOL(ACETAMINOPHEN)+PS...,HIKMA PHARMA,COLD PRODUCTS,ORAL.SOLID,64.0
2,1 2 3 (ONE TWO THREE) SUSP. 120 ML,1 2 3,CHLORPHENIRAMINE+PARACETAMOL(ACETAMINOPHEN)+PS...,HIKMA PHARMA,COLD PRODUCTS,ORAL.LIQUID,7.0
3,2HC F.C.T 20 TABLETS,2هك ف.ك.ت,VITAMIN C 1 GM,ORGANIX > NOVACURE,VITAMIN C ANTIOXIDANT,ORAL.SOLID,37.0
4,2M WHITES BEEGU MARIN SPRAY,2م وهيتيس بيجو مارين,SEA WATER+EUCALYPTUS+GLYCERIN+HYALURONIC ACID,NILL FACTORY COSMETICS > 2M WHITES PHARMA,SOOTHING & MOISTURIZING SPRAY,UNKNOWN,32.0
5,3 FLY 1200 MG 20 TABS.,3 فلي,PALMITOYLETHANOLAMIDE(PEA),COPAD > DRUG PHARMA EGYPT,NEUROPATHIC PAIN,ORAL.SOLID,26.0
6,3 FLY 300 MG 20 TABS.,3 فلي,PALMITOYLETHANOLAMIDE(PEA),COPAD > DRUG PHARMA EGYPT,NEUROPATHIC PAIN,ORAL.SOLID,36.0
7,3 FLY 400 MG 20 TABS,3 فلي,PALMITOYLETHANOLAMIDE,COPAD EGYPT > DRUG PHARMA EGYPT,MANAGEMENT OF NEUROPATHIC PAIN,ORAL.SOLID,65.0
8,3 FLY 600 MG 20 TABS.,3 فلي,PALMITOYLETHANOLAMIDE(PEA),COPAD > DRUG PHARMA EGYPT,TABLET FOR NEUROPATHIC PAIN,ORAL.SOLID,32.0
9,4 WET INTIMATE GEL 100 ML,4 ويت إنتيماتي,NaN,TURKUAZ MEDICAL CO. > BROTHER PHARMA,TOPICAL LUBRICANT,TOPICAL,40.0


In [22]:
df_local.info()

<class 'pandas.DataFrame'>
RangeIndex: 25070 entries, 0 to 25069
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   commercial_name_en  25070 non-null  str    
 1   commercial_name_ar  25070 non-null  str    
 2   scientific_name     22762 non-null  str    
 3   manufacturer        24999 non-null  str    
 4   drug_class          24973 non-null  str    
 5   route               25070 non-null  str    
 6   price_egp           25070 non-null  float64
dtypes: float64(1), str(6)
memory usage: 4.7 MB


In [23]:
df_local.columns

Index(['commercial_name_en', 'commercial_name_ar', 'scientific_name',
       'manufacturer', 'drug_class', 'route', 'price_egp'],
      dtype='str')

#### connect local data with APIs


In [24]:
# Function to get local drug information
def get_local_drug_info(drug_name):
    drug_name = drug_name.strip().lower()
    # search for the drug in both English and Arabic commercial names
    match = df_local[
        df_local['commercial_name_en'].str.lower().str.contains(drug_name, na=False) |
        df_local['commercial_name_ar'].str.contains(drug_name, na=False)
    ]
    
    if not match.empty:
        row = match.iloc[0]
        return {
            "found_locally": True,
            "commercial_name": row['commercial_name_en'],
            "scientific_name": row.get('scientific_name', 'Unknown'),
            "price": row.get('price_egp', 'N/A'),
            "manufacturer": row.get('manufacturer', 'N/A')
        }
    return {"found_locally": False, "scientific_name": drug_name}

In [25]:
# 3. Global safety data fetching function from openFDA (updated for maximum safety details)
def fetch_global_safety_data(scientific_name):
    clean_name = scientific_name.split(',')[0].strip()
    url = f"https://api.fda.gov/drug/label.json?search=active_ingredient:{clean_name}&limit=1"
    
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            result = data['results'][0]
            
            # Extract available fields accurately (with fallback if a field is missing)
            warnings = " ".join(result.get('warnings', ['']))
            contraindications = " ".join(result.get('contraindications', ['']))
            boxed_warning = " ".join(result.get('boxed_warning', ['']))
            side_effects = " ".join(result.get('adverse_reactions', ['']))
            
            # Combine available texts to ensure no empty messages appear
            combined_safety_text = f"Warnings: {warnings} | Contraindications: {contraindications} | Boxed Warning: {boxed_warning}"
            
            return {
                "status": "Success",
                "warnings": warnings[:600] if warnings else "No explicit text warnings recorded.",
                "contraindications": contraindications[:600] if contraindications else "No explicit contraindications recorded.",
                "side_effects": side_effects[:600] if side_effects else "Reactions monitored within general drug warnings.",
                "full_text": combined_safety_text.lower()
            }
        else:
            return {"status": "Not_Found", "message": "Detailed global safety data not found for this drug."}
    except Exception as e:
        return {"status": "Error", "message": str(e)}

In [26]:
# 4.  (Dynamic Clinical Rule Engine)
def clinical_safety_validator(drug_query, patient_profile):
    print(f" [check drug]: {drug_query} |  [patient profile]: chronic diseases recorded: {patient_profile.get('chronic_diseases', [])}")
  
    
    # search for the drug in the local Egyptian database
    local_info = get_local_drug_info(drug_query)
    active_ingredient = local_info['scientific_name']
    
    print(f" [active ingredient]: {active_ingredient}")
    
    # extract global safety data from openFDA for the active ingredient
    safety_data = fetch_global_safety_data(active_ingredient)
    
    # Dynamic Clinical Rule Engine: Check for contraindications based on patient profile
    alerts = []                                                             # ----> just a prototype phase , next we make a database 
    chronic_diseases = patient_profile.get('chronic_diseases', [])
    
    if safety_data['status'] == "Success":
        # Combine warnings, contraindications, and side effects into a single text for easier searching
        full_text = (
            safety_data.get('warnings', '') + " " + 
            safety_data.get('contraindications', '') + " " + 
            safety_data.get('side_effects', '')
        ).lower()
        
        # check each chronic disease against the combined safety text
        for disease in chronic_diseases:
            disease_clean = disease.strip().lower()
            # check if the disease is mentioned in the safety data
            if disease_clean in full_text:
                alerts.append(f"🔴 [take care - Contraindication]: the drug has contraindications for: ({disease})!")
        
        # if no alerts were generated, it means the drug is considered safe for the given chronic diseases
        if not alerts:
            alerts.append(" the drug is considered safe for the patient's chronic diseases based on global safety data.")
    else:
        alerts.append(" No global safety data found for this drug in openFDA, please consult a healthcare professional for further guidance.")
        
        
    #print the alerts and safety summary
    print("\n the alerts satisfied for the patient's chronic diseases:")
    for alert in alerts:
        print(f"   {alert}")
        
    if safety_data['status'] == "Success":
        print(f"\  summary effects(Side Effects) (Global): \n{safety_data['side_effects']}")
    



In [27]:
# ==========================================
# Example usage of the clinical_safety_validator function
# ==========================================
dynamic_patient = {
    "patient_id": "P-505",
    "chronic_diseases": ["Asthma", "Diabetes"]  # 
}

clinical_safety_validator("Ibuprofen", dynamic_patient)

 [check drug]: Ibuprofen |  [patient profile]: chronic diseases recorded: ['Asthma', 'Diabetes']
 [active ingredient]: IBUPROFEN

 the alerts satisfied for the patient's chronic diseases:
   🔴 [take care - Contraindication]: the drug has contraindications for: (Asthma)!
\  summary effects(Side Effects) (Global): 
Reactions monitored within general drug warnings.
